# The intersect3 problem

## Statement
> Compute the intersection of **three** sets of (distinguished) positive integers. 

> For a typical problem, each of the three sets contains less than 1000 integers. All the integers are less than $10^8$. 

## Background
> The problem comes from the implementation of k-nearest-neighbor algorithm for a set of three-dimensional mesh points of size N. Such algorithm is a key component in the 3D mesh generation algorithm for finite element analysis. Naive implementations are usually very slow, because to find the k-nearest neighbors for a mesh point, the  Euclidean distances to all the N-1 neighbors are calculated. 

> The simplest improvement is to draw a cube at one mesh point and select all the mesh points within the cube. If the density of mesh points are uniform, then the number of points inside the cube would be enough for the k-nearest neighbors. The Euclidean distance computation and sorting can be limited within this cube to speed up the k-nearest neighbor search. For a 3D mesh of $N$ points,  drawing a cube would reduce the number of distance calculations by a factor of $O(N/m)$ where $m$ is the number of mesh point inside the cube. That is a great improvement because the cube typically contains no more than 1000 mesh points.          

________________

## Convenient implementation
> Use the build-in Julia funcition `intersect`. This function is actually [not trivial at all](https://github.com/JuliaLang/julia/issues/13675). A quick look at the implementation of this function in the file `array.jl:2625` and `abstractset.jl:128` (Julia Version 1.7.2 (2022-02-06)), I found that this function rely on the `Set` data structure. 

In [1]:
intersect3_v1(A,B,C) = intersect(intersect(A,B),C)

intersect3_v1 (generic function with 1 method)

> There are discussions 
> [here](https://discourse.julialang.org/t/intersect-is-slow-for-large-arrays/442/3)
> and 
> [here](https://github.com/JuliaLang/julia/issues/13675) 
> pointing out the performance bug of this build-in function. 
> Based on the discussions there, and the source code mentioned earlier, 
> I sort the length of three sets and put the smallest set to the first argument. 
> Benchmark results show that it indeed improves the performance. 

In [2]:
function intersect3_v2(A,B,C)
    lenA,lenB,lenC = length(A),length(B),length(C)
    if lenA <= lenB
        if lenB <= lenC
            return intersect(intersect(A,B),C)
        else
            return (lenA <= lenC ? intersect(intersect(A,C),B) 
                                 : intersect(intersect(C,A),B))
        end
    else # lenB < lenA
        if lenA <= lenC
            return intersect(intersect(B,A),C)
        else
            return (lenB <= lenC ? intersect(intersect(B,C),A)
                                 : intersect(intersect(C,B),A))
        end
    end
end

intersect3_v2 (generic function with 1 method)

In [3]:
using BenchmarkTools

In [4]:
b1 = @benchmarkable intersect3_v1(A,B,C)  setup=(
    A = unique(rand(1:3000000,rand(100:90000))); 
    B = unique(rand(1:3000000,rand(100:90000)));
    C = unique(rand(1:3000000,rand(100:90000))))
tune!(b1)
run(b1)

BenchmarkTools.Trial: 495 samples with 1 evaluation.
 Range (min … max):  384.902 μs … 9.377 ms  ┊ GC (min … max): 0.00% … 3.67%
 Time  (median):       4.605 ms             ┊ GC (median):    0.00%
 Time  (mean ± σ):     4.588 ms ± 2.028 ms  ┊ GC (mean ± σ):  0.59% ± 2.48%

         ▁▁▂ ▂▄▂▁▁▁▅   ▂  ▁▁▄▄▃█▄▄▅▁▁▂  ▁▅▁ ▁▃  ▁ ▃▁▁          
  ▃▁▃▆▇▄▃███▄████████▇▆███████████████▆▇███▇██▇██▆████▄▅▃▃▃▅▃ ▅
  385 μs          Histogram: frequency by time        8.79 ms <

 Memory estimate: 389.82 KiB, allocs estimate: 35.

In [5]:
b2 = @benchmarkable intersect3_v2(A,B,C)  setup=(
    A = unique(rand(1:3000000,rand(100:90000))); 
    B = unique(rand(1:3000000,rand(100:90000)));
    C = unique(rand(1:3000000,rand(100:90000))))
tune!(b2)
run(b2)

BenchmarkTools.Trial: 574 samples with 1 evaluation.
 Range (min … max):  121.046 μs … 8.649 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):       3.001 ms             ┊ GC (median):    0.00%
 Time  (mean ± σ):     3.238 ms ± 1.652 ms  ┊ GC (mean ± σ):  0.82% ± 3.52%

           ▂▄█▃ ▆▆▃▆▁▇▃▅▃▂▂▆▂▆▃▄▂   ▁ ▂ ▁                      
  ▃▃▃▄▆▇█▇████████████████████████▇▇█▇███▇▆▆▅▄▃▃▃▄▄▄▆▆▄▁▃▁▃▃▄ ▅
  121 μs          Histogram: frequency by time        7.69 ms <

 Memory estimate: 188.30 KiB, allocs estimate: 33.

> Later in this note, the benchmark results tell us that this is the fastest implementation.

________________

## Counting, when maximum of A,B,C is known
> When we are guaranteed that `A,B,C` contain integers no greater than `N`, we may simply count the number of occurances of each element in these sets and select the elements that occur exactly three times. For small values of `N`, the following simple counting method may be faster.
> The following implementation `intersect3_v3` is simple enough but the complexity is $O(N)$. 
> When the three sets `A,B,C` is small and sparse in the range `1:N`, 
> the perfornamce is far from optimal.  

In [6]:
function intersect3_v3(A,B,C; N=1_000_000)
    cnt = zeros(Int,N)
    cnt[A] .+= 1
    cnt[B] .+= 1
    cnt[C] .+= 1
    return findall(cnt.==3)
end

intersect3_v3 (generic function with 1 method)

> The benchmark is done for large and small values of `N`. For large value `N=3000000`, I find

In [7]:
b3 = @benchmarkable intersect3_v3(A,B,C;N=3000000)  setup=(
    A = unique(rand(1:3000000,rand(100:90000))); 
    B = unique(rand(1:3000000,rand(100:90000)));
    C = unique(rand(1:3000000,rand(100:90000))))
tune!(b3)
run(b3)

BenchmarkTools.Trial: 299 samples with 1 evaluation.
 Range (min … max):   3.623 ms … 58.241 ms  ┊ GC (min … max): 0.00% … 93.65%
 Time  (median):     10.721 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   10.247 ms ±  3.558 ms  ┊ GC (mean ± σ):  4.67% ±  8.57%

                                     ▂▆▃▄█▄█▄                  
  ▂▁▃▁▁▂▄▆▄▄▆▅▄▃▁▃▂▂▁▂▃▃▃▃▁▁▁▁▁▁▁▂▄████████████▅▄▃▁▁▁▁▂▁▁▁▂▁▂ ▃
  3.62 ms         Histogram: frequency by time        14.6 ms <

 Memory estimate: 23.39 MiB, allocs estimate: 12.

> For small value `N=10000`, the result for `intersect3_v3` is much better:

In [8]:
b2 = @benchmarkable intersect3_v2(A,B,C)  setup=(
    A = unique(rand(1:10000,rand(100:9000))); 
    B = unique(rand(1:10000,rand(100:9000)));
    C = unique(rand(1:10000,rand(100:9000))))
tune!(b2)
run(b2)

BenchmarkTools.Trial: 6243 samples with 1 evaluation.
 Range (min … max):   27.564 μs …   1.444 ms  ┊ GC (min … max): 0.00% … 49.48%
 Time  (median):     283.142 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   311.868 μs ± 165.407 μs  ┊ GC (mean ± σ):  1.22% ±  4.71%

        ▁▃▆▆█▇▆▅▇█▇▄▄▅▆▄▄▅▂▂▂▂▃▃▁▁▂ ▂▁▂                          
  ▁▂▃▄▆▇█████████████████████████████████▆▆▇▅▆▄▄▅▃▄▃▄▃▃▃▃▃▂▂▃▂▂ ▅
  27.6 μs          Histogram: frequency by time          730 μs <

 Memory estimate: 21.68 KiB, allocs estimate: 33.

In [9]:
b3 = @benchmarkable intersect3_v3(A,B,C;N=10000)  setup=(
    A = unique(rand(1:10000,rand(100:9000))); 
    B = unique(rand(1:10000,rand(100:9000)));
    C = unique(rand(1:10000,rand(100:9000))))
tune!(b3)
run(b3)

BenchmarkTools.Trial: 9363 samples with 1 evaluation.
 Range (min … max):  14.100 μs … 811.811 μs  ┊ GC (min … max): 0.00% … 87.50%
 Time  (median):     40.759 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   44.257 μs ±  50.822 μs  ┊ GC (mean ± σ):  7.51% ±  6.25%

                  ▂▂▄▄▅▆▆▇▆█▇█▆▇▆▄▃▂▁                           
  ▁▁▁▂▂▂▂▂▃▃▃▅▆▇██████████████████████▆▅▄▃▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▄
  14.1 μs         Histogram: frequency by time         77.4 μs <

 Memory estimate: 93.41 KiB, allocs estimate: 9.

________________

## Exploit the orderedness of integer set
> The integer sets can be sorted. With sorted lists $A,B,C$, 
> the intersection algorithm can be faster because,
> when we build the set $I = A\cap B\cap C$, 
> each time we reject an element, say $a \in A$, from being added into $I$, 
> we can jump in the the other two sorted lists to skip the elements 
> which are smaller than $a$. 

> The following version is a fakakta implementation based on this idea. 

In [10]:
function bisec(a::Int, b::Int, x::Int, Lref::Base.RefValue{Vector{Int64}})::Int
    (x <  Lref[][a]) && (return a-1)
    (x >= Lref[][b]) && (return b)
    c = 0
    while b - a > 1
        c = (a + b) ÷ 2
        if x < Lref[][c]
            b = c
        else
            a = c
        end
    end
    x < Lref[][b] ? a : b
end

bisec (generic function with 1 method)

In [11]:
function ord3!(ord,s,t)
    if s[1]-t[1] <= s[2]-t[2]
        return (s[2]-t[2] <= s[3]-t[3] 
                    ? (ord[1]=1; ord[2]=2; ord[3]=3;)
                    : (s[1]-t[1] <= s[3]-t[3] 
                            ? (ord[1]=1;ord[2]=3;ord[3]=2;) 
                            : (ord[1]=3;ord[2]=1;ord[3]=2;)))
    else # s[2] < s[1]
        return (s[1]-t[1] <= s[3]-t[3] 
                    ? (ord[1]=2;ord[2]=1;ord[3]=3;)
                    : (s[2]-t[2] <= s[3]-t[3] 
                            ? (ord[1]=2;ord[2]=3;ord[3]=1;) 
                            : (ord[1]=3;ord[2]=2;ord[3]=1;)))
    end    
end

allposv(v) = v[1]>0 && v[2]>0 && v[3]>0

function intersect3_v4(A,B,C)
    # profiling shows that the major efforts are sorting
    As = sort(A)
    Bs = sort(B)
    Cs = sort(C)
    # use reference to avoid allocations
    Lref = [Ref(As),Ref(Bs),Ref(Cs)] ;
    # preparations
    maxt = maximum(r->first(r[]), Lref)
    mint = minimum(r->last(r[]), Lref)
    head = [bisec(1,length(r[]),maxt,r) for r in Lref]
    tail = [bisec(1,length(r[]),mint,r) for r in Lref]
    s = [-1,-1,-1]  #+ reduce alloc
    ord3!(s, tail, head)  #+ reduce alloc
    if !(allposv(head) && head[s[1]]<=tail[s[1]])
        return []
    end
    intsct = zeros(Int,minimum(tail.-head))
    p = 1
    while allposv(head) && head[s[1]]<=tail[s[1]]
        ord3!(s, tail, head)  #+ reduce alloc
        a = Lref[s[1]][][head[s[1]]]
        head[s[1]] = head[s[1]]+1
        # search in L[head:tail]
        i2 = bisec(head[s[2]], tail[s[2]], a, Lref[s[2]])
        if i2 < head[s[2]]
            # outside range head:tail
            continue
        elseif a != Lref[s[2]][][i2]
            head[s[2]] = i2+1
            continue
        else # a == Lref[s[2]][][i2]
            i3 = bisec(head[s[3]], tail[s[3]], a, Lref[s[3]])
            if i3 < head[s[3]]
                head[s[2]] = i2+1
                continue
            else
                if a == Lref[s[3]][][i3]
                    intsct[p] = a
                    p += 1
                end
                head[s[2]] = i2+1
                head[s[3]] = i3+1
                continue
            end
        end
    end
    intsct[1:(p-1)]
end


intersect3_v4 (generic function with 1 method)

> Of course, fakakta does not imply broken. This implementation is at least correct:

In [12]:
for i=1:10000
    A = unique(rand(1:3000,rand(300:5000)))
    B = unique(rand(1:1000,rand(300:5000)))
    C = unique(rand(1:6000,rand(300:5000)))
    @assert sort(intersect3_v2(A,B,C))==intersect3_v4(A,B,C) 
end

> Now we do benchmark to compare `intersect3_v4` against `intersect3_v2`:

> Small size:

In [13]:
b2 = @benchmarkable intersect3_v2(A,B,C)  setup=(
    A = unique(rand(1:10000,rand(100:9000))); 
    B = unique(rand(1:10000,rand(100:9000)));
    C = unique(rand(1:10000,rand(100:9000))))
tune!(b2)
run(b2)

BenchmarkTools.Trial: 6273 samples with 1 evaluation.
 Range (min … max):   27.600 μs …   1.501 ms  ┊ GC (min … max): 0.00% … 49.73%
 Time  (median):     279.286 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   310.452 μs ± 169.007 μs  ┊ GC (mean ± σ):  1.56% ±  5.45%

        ▁▃▆███▅▆█▆▅▅▃▃▃▄▁▃▁▃▃▂▁▁▁▂▁▂▃▁                           
  ▁▂▃▄▅████████████████████████████████▇▅▆▆▅▅▄▄▄▄▃▄▃▃▃▃▃▃▂▂▂▂▂▂ ▅
  27.6 μs          Histogram: frequency by time          748 μs <

 Memory estimate: 28.02 KiB, allocs estimate: 33.

In [14]:
b4 = @benchmarkable intersect3_v4(A,B,C)  setup=(
    A = unique(rand(1:10000,rand(100:9000))); 
    B = unique(rand(1:10000,rand(100:9000)));
    C = unique(rand(1:10000,rand(100:9000))))
tune!(b4)
run(b4)

BenchmarkTools.Trial: 4997 samples with 1 evaluation.
 Range (min … max):   51.621 μs …   1.848 ms  ┊ GC (min … max): 0.00% … 40.60%
 Time  (median):     496.460 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   509.845 μs ± 189.231 μs  ┊ GC (mean ± σ):  0.47% ±  3.16%

                 ▁▂▃▃█▅█▇▅▇▇▇▇▇▇▆█▆▆▅▄▆▆▇▃▁▃▃▂▂▂                 
  ▁▂▂▂▂▂▃▄▄▄▄▅▅████████████████████████████████████▇▆▇▆▆▅▄▃▄▃▃▃ ▆
  51.6 μs          Histogram: frequency by time          935 μs <

 Memory estimate: 14.08 KiB, allocs estimate: 13.

> Large size:

In [15]:
b2 = @benchmarkable intersect3_v2(A,B,C)  setup=(
    A = unique(rand(1:3000000,rand(100:90000))); 
    B = unique(rand(1:3000000,rand(100:90000)));
    C = unique(rand(1:3000000,rand(100:90000))))
tune!(b2)
run(b2)

BenchmarkTools.Trial: 559 samples with 1 evaluation.
 Range (min … max):  222.843 μs … 8.840 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):       3.093 ms             ┊ GC (median):    0.00%
 Time  (mean ± σ):     3.391 ms ± 1.700 ms  ┊ GC (mean ± σ):  0.82% ± 3.00%

           ▅▁▄▃▃▆▁█▇▆▆█▄▆▃▄▅ ▂ ▁   ▃ ▄▂                        
  ▄▅▄▆▄▇█▅▇███████████████████▇█▇▆▇█▆██▅▆█▆▅█▆▄▇▃▄▅▃▃▃▄▃▄▃▁▁▃ ▅
  223 μs          Histogram: frequency by time        7.93 ms <

 Memory estimate: 205.87 KiB, allocs estimate: 33.

In [16]:
b4 = @benchmarkable intersect3_v4(A,B,C)  setup=(
    A = unique(rand(1:3000000,rand(100:90000))); 
    B = unique(rand(1:3000000,rand(100:90000)));
    C = unique(rand(1:3000000,rand(100:90000))))
tune!(b4)
run(b4)

BenchmarkTools.Trial: 395 samples with 1 evaluation.
 Range (min … max):  764.488 μs … 14.880 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):       7.295 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):     7.332 ms ±  2.651 ms  ┊ GC (mean ± σ):  0.20% ± 1.07%

                 ▁    ▁  ▁▅▁ ▂▁▅▁▅ █ ▃▁   ▁                     
  ▂▂▂▂▃▃▂▂▂▄▃▄▂▅▃█▄▄▆▆█▆▇███▆███████▆███▇██▅▂▅▄▄▄▄▄▄▄▂▃▃▂▃▂▂▂▂ ▃
  764 μs          Histogram: frequency by time         13.6 ms <

 Memory estimate: 155.88 KiB, allocs estimate: 15.

> The following implementation `` assumes that the input lists are sorted:

In [17]:
# for sorted integer lists
function intersect3_v4s(As,Bs,Cs)
    # use reference to avoid allocations
    Lref = [Ref(As),Ref(Bs),Ref(Cs)] ;
    # preparations
    maxt = maximum(r->first(r[]), Lref)
    mint = minimum(r->last(r[]), Lref)
    head = [bisec(1,length(r[]),maxt,r) for r in Lref]
    tail = [bisec(1,length(r[]),mint,r) for r in Lref]
    s = [-1,-1,-1]  #+ reduce alloc
    ord3!(s, tail, head)  #+ reduce alloc
    if !(allposv(head) && head[s[1]]<=tail[s[1]])
        return []
    end
    intsct = zeros(Int,minimum(tail.-head))
    p = 1
    while allposv(head) && head[s[1]]<=tail[s[1]]
        ord3!(s, tail, head)  #+ reduce alloc
        a = Lref[s[1]][][head[s[1]]]
        head[s[1]] = head[s[1]]+1
        # search in L[head:tail]
        i2 = bisec(head[s[2]], tail[s[2]], a, Lref[s[2]])
        if i2 < head[s[2]]
            # outside range head:tail
            continue
        elseif a != Lref[s[2]][][i2]
            head[s[2]] = i2+1
            continue
        else # a == Lref[s[2]][][i2]
            i3 = bisec(head[s[3]], tail[s[3]], a, Lref[s[3]])
            if i3 < head[s[3]]
                head[s[2]] = i2+1
                continue
            else
                if a == Lref[s[3]][][i3]
                    intsct[p] = a
                    p += 1
                end
                head[s[2]] = i2+1
                head[s[3]] = i3+1
                continue
            end
        end
    end
    intsct[1:(p-1)]
end


intersect3_v4s (generic function with 1 method)

> You can try out the benchmark yourself to see the performance improvement of `intersect3_v4s`.
> To be fair, we use the sorted lists for all benchmark cases.

In [18]:
b4s = @benchmarkable intersect3_v4s(As,Bs,Cs)  setup=(
    As = sort(unique(rand(1:3000000,rand(100:90000)))); 
    Bs = sort(unique(rand(1:3000000,rand(100:90000))));
    Cs = sort(unique(rand(1:3000000,rand(100:90000)))))
tune!(b4s)
run(b4s)

BenchmarkTools.Trial: 378 samples with 1 evaluation.
 Range (min … max):   13.701 μs …   2.900 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     896.652 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   966.388 μs ± 613.482 μs  ┊ GC (mean ± σ):  0.13% ± 1.04%

     ▃▇▄▅▇▃▄▄ ▃▃▇▃▃▄ █▄▆▂▇▄▃▄█▂   ▅▅ ▂     ▂                     
  █▇█████████▅██████▅██████████▄█▇██▆██▆█▆▇█▁▅▇▄▇▅▅▅▄▃▅▃▄▃█▁▅▄▄ ▅
  13.7 μs          Histogram: frequency by time         2.42 ms <

 Memory estimate: 1.83 KiB, allocs estimate: 10.

In [19]:
b4 = @benchmarkable intersect3_v4(As,Bs,Cs)  setup=(
    As = sort(unique(rand(1:3000000,rand(100:90000)))); 
    Bs = sort(unique(rand(1:3000000,rand(100:90000))));
    Cs = sort(unique(rand(1:3000000,rand(100:90000)))))
tune!(b4)
run(b4)

BenchmarkTools.Trial: 361 samples with 1 evaluation.
 Range (min … max):  209.007 μs …   4.375 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):       1.540 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):     1.733 ms ± 910.550 μs  ┊ GC (mean ± σ):  0.48% ± 2.37%

       ▂▂▄▅ ▆█▄ ▅▃▇▂▂▃▃▅▆▃▂▅▁  ▂▁            ▃                   
  ▃▁▇█▄████▅███▆█████████████▇▅██▅▇▅▇███▇██▇▆█▆▄█▄▅▃▇▃▃▅▆▃▄▇▃▃▆ ▅
  209 μs           Histogram: frequency by time         3.82 ms <

 Memory estimate: 160.44 KiB, allocs estimate: 15.

In [20]:
b2 = @benchmarkable intersect3_v2(As,Bs,Cs)  setup=(
    As = sort(unique(rand(1:3000000,rand(100:90000)))); 
    Bs = sort(unique(rand(1:3000000,rand(100:90000))));
    Cs = sort(unique(rand(1:3000000,rand(100:90000)))))
tune!(b2)
run(b2)

BenchmarkTools.Trial: 323 samples with 1 evaluation.
 Range (min … max):  324.405 μs … 8.656 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):       2.973 ms             ┊ GC (median):    0.00%
 Time  (mean ± σ):     3.287 ms ± 1.592 ms  ┊ GC (mean ± σ):  0.75% ± 2.97%

             █▆▇  ▇▁▆▃▄▇ ▅▁   ▂▁   ▁▁ ▁ ▁ ▂                    
  ▆▄▁▃▆▇▄▆▅█▇████▆██████████▇███▆█▇██▄█▅█▆█▄▅▅▅▃▇▅▄▅▄▄▃▁▅▃▃▃▄ ▅
  324 μs          Histogram: frequency by time        7.26 ms <

 Memory estimate: 312.52 KiB, allocs estimate: 33.

## Conclusion
> The best algorithm for intersect3 problem depends on the scales:
> 1. for sets with elements less than $N \sim 10^4$, a direct counting is simple and fast;
> 2. for sets with elements less than $N \sim 10^8$, the Julia implementation wins;
> 3. if the sets are given as sorted lists, the fakakta implementation `intersect3_v4s` is the best.